# 🎙️ Treinamento da Voz Oficial do Mascote FEFO (Fine-Tuning F5-TTS)
**Projeto FEFO** — Notebook 100% automatizado para treinar uma IA dedicada com as gravações oficiais do FEFO (577 frases / 25 minutos de áudio).

---
### ⚡ Instruções (Apenas 1 Clique):
1. Verifique se a GPU está ativa: No menu superior, vá em **Ambiente de execução** ➔ **Alterar tipo de ambiente de execução** ➔ Selecione **T4 GPU** ➔ **Salvar**.
2. Clique em **Ambiente de execução** ➔ **Executar tudo** (ou pressione `Ctrl + F9`).
3. O notebook vai carregar o dataset, treinar o modelo (~45 min) e salvar os pesos treinados automaticamente no GitHub e para download!

In [ ]:
# 1. Verificação da GPU NVIDIA
import torch
print('=' * 60)
if torch.cuda.is_available():
    print(f'✓ GPU DETECTADA: {torch.cuda.get_device_name(0)}')
    print(f'✓ VRAM Disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada!')
    print('Vá em: Ambiente de execução -> Alterar tipo de ambiente -> Escolha T4 GPU.')
print('=' * 60)

In [ ]:
# 2. Clonagem do Repositório FEFO e Download do Dataset
%cd /content
!git clone https://github.com/elCortelini/fefo-v1.git /content/fefo-repo
print('✓ Repositório FEFO clonado com sucesso!')

In [ ]:
# 3. Instalação do F5-TTS e Dependências de Áudio
%cd /content
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -q -e .
!pip install -q huggingface_hub vocos soundfile torchaudio accelerate pydub gdown
print('✓ Dependências instaladas!')

In [ ]:
# 4. Extração do Dataset do FEFO (577 frases / 25 minutos de áudio)
import os, zipfile

dataset_dir = '/content/F5-TTS/data/fefo'
os.makedirs(dataset_dir, exist_ok=True)

zip_source = '/content/fefo-repo/fefo_training_dataset.zip'
if not os.path.exists(zip_source):
    # Fallback se não estiver no repo
    from google.colab import files
    print('Envie o arquivo fefo_training_dataset.zip do seu computador:')
    uploaded = files.upload()
    zip_source = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_source, 'r') as z:
    z.extractall(dataset_dir)

print(f'✓ Dataset extraído com sucesso em: {dataset_dir}')
!head -n 5 {dataset_dir}/metadata.csv

In [ ]:
# 5. Estruturação dos Metadados para Treino
!python src/f5_tts/train/datasets/prepare_csv_wavs.py \
    --csv_file {dataset_dir}/metadata.csv \
    --wavs_dir {dataset_dir}/wavs \
    --output_dir {dataset_dir}
print('✓ Dados estruturados com sucesso!')

In [ ]:
# 6. Execução do Treinamento (Fine-Tuning com Pesos em Português do Brasil)
# Treina por 2000 passos (cerca de 40 a 50 minutos em GPU T4)
!python src/f5_tts/train/train.py \
    --exp_name fefo_voice_model \
    --learning_rate 1e-5 \
    --batch_size_per_gpu 4 \
    --batch_size_type frame \
    --max_samples 64 \
    --grad_accumulation_steps 2 \
    --max_steps 2000 \
    --save_per_steps 500 \
    --finetune True \
    --pretrain firstpixel/F5-TTS-pt-br \
    --dataset_name fefo

In [ ]:
# 7. Teste de Síntese com a Voz Dedicada do FEFO
import os, glob, IPython.display as ipd

ckpts = glob.glob('/content/F5-TTS/ckpts/fefo_voice_model/*.safetensors')
ckpt_file = sorted(ckpts)[-1] if ckpts else '/content/F5-TTS/ckpts/fefo_voice_model/model_last.safetensors'
print(f'Usando checkpoint: {ckpt_file}')

test_phrase = 'Olá amiguinho! Eu sou o Fefo, vamos brincar juntos hoje? Estou muito feliz em te ver!'
ref_audio = f'{dataset_dir}/wavs/fefo_0001.wav'
ref_text = 'fico feliz por você'

!python src/f5_tts/infer/infer_cli.py \
    --model 'F5-TTS' \
    --ckpt_file '{ckpt_file}' \
    --vocab_file 'src/f5_tts/infer/examples/vocab.txt' \
    --ref_audio '{ref_audio}' \
    --ref_text '{ref_text}' \
    --gen_text '{test_phrase}' \
    --output_dir '/content/teste_fefo'

print('✓ ÁUDIO DO FEFO GERADO COM O MODELO TREINADO! OUÇA ABAIXO:')
ipd.Audio('/content/teste_fefo/out.wav')

In [ ]:
# 8. Salvamento e Download do Modelo Treinado
from google.colab import files
import shutil
import os

final_model_dest = '/content/fefo_model.safetensors'
if os.path.exists(ckpt_file):
    shutil.copyfile(ckpt_file, final_model_dest)
    print(f'✓ Modelo copiado para {final_model_dest}')
else:
    print(f'Checkpoint não encontrado em {ckpt_file}')

print('=' * 60)
print('🎉 TREINAMENTO CONCLUÍDO COM SUCESSO!')
print('Iniciando download automático de fefo_model.safetensors...')
print('Basta colocar o arquivo baixado na pasta models/ do projeto FEFO!')
print('=' * 60)
try:
    files.download(final_model_dest)
except Exception as e:
    print(f'Aviso download do navegador: {e}')

# (Opcional) Envio direto para o GitHub se desejar sincronizar remotamente
gh_token = ''
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    gh_token = ''

if not gh_token:
    print('\n[Opcional] Para salvar direto no repositório GitHub:')
    gh_token = input('Cole seu Personal Access Token do GitHub (ou pressione ENTER para pular): ').strip()

if gh_token:
    try:
        %cd /content/fefo-repo
        !mkdir -p models
        !cp /content/fefo_model.safetensors models/fefo_model.safetensors
        !git config user.name 'FEFO Voice Trainer'
        !git config user.email 'fefo@bot.com'
        !git add models/fefo_model.safetensors
        !git commit -m 'feat(ai): modelo oficial treinado do FEFO (2000 passos F5-TTS)'
        !git push https://{gh_token}@github.com/elCortelini/fefo-v1.git main
        print('✓ Modelo salvo diretamente no repositório GitHub!')
    except Exception as e:
        print(f'Aviso push GitHub: {e}')
